# Topic 3: Web Scraping

## Extracting Structured Data from the Web

This notebook is the practical companion for Topic 3 of the Unstructured Data module. It consolidates all web scraping concepts into a single, progressive learning journey — from making your first HTTP request through to building a respectful, reusable scraping pipeline.

### How This Notebook Is Organised

The notebook is divided into **in-lesson** and **independent study** sections:

| Section | Mode | What You Will Learn |
|---------|------|-------------------|
| Part 1: How the Web Works | 🏫 In-lesson | HTTP requests, HTML parsing with BeautifulSoup |
| Part 2: Extracting Structured Data | 🏫 In-lesson | Scraping a product catalogue, building a DataFrame |
| Part 3: Ethical & Respectful Scraping | 🏫 In-lesson | robots.txt, crawl delays, legal considerations |
| Part 4: Bringing It Together | 🏫 In-lesson | Tool comparison, scraping → preprocessing preview |
| Part 5: Dynamic Websites with Selenium | 📖 Independent study | JavaScript-rendered pages, headless browsers |
| Part 6: From Scraping to Insight | 📖 Independent study | End-to-end pipeline: scrape → preprocess → sentiment |
| Exercises | 📖 Independent study | Assessment-aligned consolidation tasks |

### Link to Your Assessment

This topic directly supports **Learning Outcome 3**: *Scrape information from external websites and process into a structured format.* The assessment rubric rewards a structured methodology, discussion of available tools, clear business value, and awareness of legal/ethical implications. Every section of this notebook connects to at least one of these criteria.

---

## Setup

Run the cell below to install the required libraries. We will use:

- **requests** — making HTTP requests to web servers
- **BeautifulSoup4** — parsing and navigating HTML documents
- **pandas** — structuring scraped data into DataFrames
- **selenium** & **webdriver-manager** — automating a browser for JavaScript-rendered pages (Part 5)

In [ ]:
%pip install --quiet beautifulsoup4 requests pandas selenium webdriver-manager

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time
import re
import logging
from urllib.robotparser import RobotFileParser
from urllib.parse import urlparse, urljoin
from typing import Optional

---

# 🏫 Part 1: How the Web Works

*Approximate time: 25 minutes*

Before we can scrape a website, we need to understand how your browser retrieves a web page. Every time you visit a URL, three things happen:

1. **Your browser sends an HTTP request** to a web server (usually a `GET` request)
2. **The server responds** with an HTTP response containing HTML (and a status code)
3. **Your browser renders the HTML** into the page you see

Web scraping automates steps 1 and 2 — we send the request programmatically, receive the HTML, and extract the data we need *without* a browser.

### Why Does This Matter for Data Scientists?

Web scraping transforms the unstructured web into structured, analysable data. Common business applications include:

- **Competitive intelligence** — monitoring competitor pricing, product ranges, or job postings
- **Market research** — aggregating customer reviews or industry news
- **Regulatory monitoring** — tracking changes to government publications or compliance data
- **Lead generation** — extracting contact information from business directories

> 💼 **Assessment connection:** The assessment brief asks you to identify a relevant business challenge where collecting and analysing unstructured data adds value. Web scraping is one of the primary collection methods you can demonstrate.

## 1.1 Making an HTTP Request

Let's start by requesting a real web page. We will use [Books to Scrape](http://books.toscrape.com/), a website specifically designed for practising web scraping techniques. Think of it as a simulated product catalogue — the kind of data source a business analyst might scrape to monitor competitor pricing.

In [ ]:
url = "http://books.toscrape.com/"

response = requests.get(url)

print(f"Status code: {response.status_code} {response.reason}")
print(f"Content type: {response.headers.get('Content-Type', 'unknown')}")
print(f"Content length: {len(response.text):,} characters")

The status code tells us whether the request was successful:

| Code | Meaning | What to Do |
|------|---------|------------|
| 200 | OK — page returned successfully | Proceed with parsing |
| 403 | Forbidden — access denied | Check robots.txt, add headers |
| 404 | Not Found — page doesn't exist | Check the URL |
| 429 | Too Many Requests — rate limited | Slow down, add delays |
| 500 | Server Error | Try again later |

Let's look at the raw HTML that was returned:

In [ ]:
# Show the first 500 characters of the HTML
print(response.text[:500])

This is what your browser receives — a wall of HTML tags and text. Our job is to parse this into something useful.

## 1.2 Parsing HTML with BeautifulSoup

BeautifulSoup turns raw HTML into a navigable tree structure. Think of it as giving you a map of the document — you can search for specific elements by their tag name, class, or attributes.

Let's parse our response:

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

# The title tag tells us what page we're on
print(f"Page title: {soup.title.get_text()}")

### Navigating the HTML Tree

Before we scrape, we need to understand the page structure. The key to effective web scraping is **inspecting the HTML** to find patterns in the elements that contain the data you want.

> 💡 **Tip:** In your browser, right-click on any element and select "Inspect" to see its HTML. This is the single most important skill in web scraping.

Let's explore the main navigation methods:

In [ ]:
# find() returns the FIRST matching element
first_link = soup.find("a")
print(f"First link: {first_link}")
print(f"Link text: {first_link.get_text()}")
print(f"Link href: {first_link.get('href')}")

In [ ]:
# find_all() returns ALL matching elements
all_links = soup.find_all("a")
print(f"Total links on page: {len(all_links)}")

# Show first 5 links
for link in all_links[:5]:
    print(f"  {link.get_text(strip=True)[:50]:50s} → {link.get('href', 'N/A')}")

In [ ]:
# Search by CSS class — this is how you target specific elements
# On books.toscrape.com, each book is in an <article> tag with class "product_pod"
products = soup.find_all("article", class_="product_pod")
print(f"Number of products on page: {len(products)}")

# Let's examine the first product
first_product = products[0]
print(f"\nFirst product HTML (truncated):\n{str(first_product)[:300]}...")

### Understanding the HTML Structure

Each product on the page follows a consistent HTML pattern. Here is a simplified version:

```html
<article class="product_pod">
    <div class="image_container">
        <a href="catalogue/a-light-in-the-attic_1000/index.html">
            <img src="media/cache/2c/da/2cdad67c44b002e7c0cddd0be...jpg"
                 alt="A Light in the Attic">
        </a>
    </div>
    <p class="star-rating Three"></p>
    <h3>
        <a href="catalogue/a-light-in-the-attic_1000/index.html"
           title="A Light in the Attic">A Light in the ...</a>
    </h3>
    <div class="product_price">
        <p class="price_color">£51.77</p>
        <p class="instock availability">
            <i class="icon-ok"></i> In stock
        </p>
    </div>
</article>
```

The pattern is clear: every product has the same structure. We can use this consistency to extract data systematically. This is the core principle of web scraping — **find the pattern, write the code to extract it**.

---

# 🏫 Part 2: Extracting Structured Data

*Approximate time: 25 minutes*

Now that we understand the basics, let's extract real data from the product catalogue and build a structured DataFrame. This mirrors a common workplace task: turning unstructured web data into something you can analyse in a spreadsheet or dashboard.

## 2.1 Extracting Product Information

We will extract four pieces of information for each book:
1. **Title** — the full book title
2. **Price** — the price in GBP
3. **Rating** — the star rating (1–5)
4. **Availability** — whether the book is in stock

In [ ]:
# Map the CSS class names to numeric ratings
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

data = []

for product in products:
    # Title: stored in the <a> tag inside <h3>, in the 'title' attribute
    title_tag = product.find("h3").find("a")
    title = title_tag["title"]

    # Price: in a <p> tag with class "price_color"
    price_text = product.find("p", class_="price_color").get_text()
    price = float(price_text.strip("£"))

    # Rating: encoded as a CSS class on the <p> tag (e.g., "star-rating Three")
    rating_tag = product.find("p", class_="star-rating")
    rating_class = rating_tag["class"][1]  # Second class is the rating word
    rating = rating_map.get(rating_class, 0)

    # Availability: in a <p> tag with class "instock"
    availability = product.find("p", class_="instock") is not None

    data.append({
        "title": title,
        "price": price,
        "rating": rating,
        "in_stock": availability
    })

# Show the first few records
data[:3]

Let's turn this into a pandas DataFrame — the standard format for structured data analysis:

In [ ]:
df = pd.DataFrame(data)

print(f"Scraped {len(df)} products from the first page")
print(f"\nPrice range: £{df['price'].min():.2f} - £{df['price'].max():.2f}")
print(f"Average rating: {df['rating'].mean():.1f} stars")

df.head(10)

## 2.2 Scraping Multiple Pages

A single page gave us 20 products, but the catalogue has 1,000 books across 50 pages. In a real business scenario, you would want the complete dataset.

The page URLs follow a predictable pattern — this is sometimes called **URL hacking**:
- Page 1: `http://books.toscrape.com/catalogue/page-1.html`
- Page 2: `http://books.toscrape.com/catalogue/page-2.html`
- Page N: `http://books.toscrape.com/catalogue/page-{N}.html`

Let's scrape the first 3 pages to demonstrate the pattern (we will keep it small to be respectful of the server):

In [ ]:
def scrape_page(url: str) -> list[dict]:
    """Scrape product data from a single page of books.toscrape.com."""
    response = requests.get(url)
    response.raise_for_status()  # Raise an exception for HTTP errors

    soup = BeautifulSoup(response.text, "html.parser")
    products = soup.find_all("article", class_="product_pod")

    page_data = []
    for product in products:
        title_tag = product.find("h3").find("a")
        price_text = product.find("p", class_="price_color").get_text()
        rating_class = product.find("p", class_="star-rating")["class"][1]

        page_data.append({
            "title": title_tag["title"],
            "price": float(price_text.strip("£Â")),
            "rating": rating_map.get(rating_class, 0),
            "in_stock": product.find("p", class_="instock") is not None
        })

    return page_data

In [ ]:
all_data = []

for page_num in range(1, 4):  # Pages 1 to 3
    url = f"http://books.toscrape.com/catalogue/page-{page_num}.html"
    print(f"Scraping {url}...")

    page_data = scrape_page(url)
    all_data.extend(page_data)

    # Be polite - wait 1 second between requests
    time.sleep(1)

df_full = pd.DataFrame(all_data)
print(f"\nTotal products scraped: {len(df_full)}")
df_full.head()

Notice the `time.sleep(1)` call between requests. This is a basic form of **rate limiting** — we are being polite to the server by not flooding it with requests. We will formalise this approach in Part 3.

> 💼 **Assessment connection:** Your coursework should demonstrate a structured methodology. Breaking your scraper into functions (like `scrape_page()`) shows good software engineering practice and makes your code reproducible — both of which are rewarded in the marking criteria.

---

# 🏫 Part 3: Ethical & Respectful Scraping

*Approximate time: 20 minutes*

Web scraping sits in a complex legal and ethical space. As data scientists — and particularly as apprentices who will apply these skills in your organisations — you need to understand the boundaries.

## 3.1 The Legal Landscape

Three areas of UK law are particularly relevant to web scraping:

**UK GDPR & Data Protection Act 2018**
If you scrape personal data (names, email addresses, user profiles), you are subject to data protection law. You must have a lawful basis for processing, and you must consider the rights of the data subjects. Scraping publicly visible data does not automatically make it lawful to process.

**Computer Misuse Act 1990**
Accessing a computer system without authorisation is a criminal offence. If a website's terms of service explicitly prohibit scraping, or if you bypass technical protections (such as CAPTCHAs or IP blocks), you may be in breach.

**Copyright, Designs and Patents Act 1988**
The content on a website — text, images, databases — is typically protected by copyright. Scraping the content for analysis is generally acceptable, but republishing or redistributing it may not be.

> ⚠️ **Key principle:** Always check the website's terms of service, robots.txt file, and whether you are collecting personal data before scraping.

## 3.2 Understanding robots.txt

The `robots.txt` file is a standard that websites use to communicate their scraping preferences to automated tools. It is located at the root of every domain (e.g., `https://example.com/robots.txt`).

Let's read a real one:

In [ ]:
# Fetch and display the robots.txt for books.toscrape.com
robots_response = requests.get("http://books.toscrape.com/robots.txt")
print(f"Status: {robots_response.status_code}")
print(f"\nContent:\n{robots_response.text}")

This is a permissive robots.txt — it does not disallow anything. Let's compare it with a more restrictive example:

In [ ]:
# Let's check a few real-world robots.txt files programmatically
sites = [
    "https://www.gov.uk/",
    "https://www.trustpilot.com/",
]

for site in sites:
    parsed = urlparse(site)
    robots_url = f"{parsed.scheme}://{parsed.hostname}/robots.txt"

    rp = RobotFileParser(robots_url)
    try:
        rp.read()
        crawl_delay = rp.crawl_delay("*") or "Not specified"
        can_fetch = rp.can_fetch("*", site)
        print(f"{parsed.hostname}")
        print(f"  Crawl delay: {crawl_delay}")
        print(f"  Can fetch homepage: {can_fetch}")
        print()
    except Exception as e:
        print(f"{parsed.hostname}: Could not read robots.txt ({e})\n")

### How to Read robots.txt

The file uses a simple format:

```
User-agent: *          # Rules apply to all bots
Disallow: /admin/      # Do not scrape anything under /admin/
Disallow: /private/    # Do not scrape anything under /private/
Crawl-delay: 10        # Wait 10 seconds between requests

User-agent: Googlebot  # Special rules for Google's crawler
Allow: /               # Google can access everything
```

Python's `urllib.robotparser` module lets us check these rules programmatically:

In [ ]:
# Check if a specific URL is allowed
rp = RobotFileParser("http://books.toscrape.com/robots.txt")
rp.read()

test_urls = [
    "http://books.toscrape.com/catalogue/page-1.html",
    "http://books.toscrape.com/index.html",
]

for test_url in test_urls:
    allowed = rp.can_fetch("*", test_url)
    print(f"{'✓' if allowed else '✗'} {test_url}")

## 3.3 Building a Respectful Scraper

Let's combine everything we have learned into a reusable `RespectfulScraper` class that automatically checks robots.txt, respects crawl delays, and uses proper logging.

This is the kind of well-engineered code that demonstrates professional practice in your assessment:

In [ ]:
class RespectfulScraper:
    """
    A web scraper that respects robots.txt rules and rate limits.

    This class automatically:
    - Parses robots.txt for the target domain
    - Enforces crawl delays between requests
    - Identifies itself with a descriptive User-Agent string
    - Logs all activity for transparency and debugging
    """

    def __init__(self, base_url: str, default_delay: int = 3):
        self.base_url = base_url.rstrip("/")

        # Create a session for connection reuse and consistent headers
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "BPP-DataScience-Bot/1.0 (educational project)"
        })

        # Set up logging
        self.logger = logging.getLogger(f"scraper.{urlparse(base_url).hostname}")
        self.logger.setLevel(logging.INFO)

        # Parse robots.txt
        self.rp = RobotFileParser()
        self.rp.set_url(f"{self.base_url}/robots.txt")

        try:
            self.rp.read()
            self.logger.info("Successfully parsed robots.txt")
        except Exception as e:
            self.logger.warning(f"Could not read robots.txt: {e}")

        # Get crawl delay: use robots.txt value, or fall back to default
        delay = self.rp.crawl_delay(self.session.headers["User-Agent"])
        self.crawl_delay = int(delay) if delay else default_delay
        self.logger.info(f"Using crawl delay of {self.crawl_delay}s")

    def can_fetch(self, url: str) -> bool:
        """Check if a URL is allowed by robots.txt."""
        return self.rp.can_fetch(self.session.headers["User-Agent"], url)

    def get(self, url: str) -> Optional[BeautifulSoup]:
        """
        Fetch and parse a page, respecting robots.txt and crawl delays.

        Returns a BeautifulSoup object if successful, None otherwise.
        """
        if not self.can_fetch(url):
            self.logger.warning(f"Blocked by robots.txt: {url}")
            return None

        try:
            self.logger.info(f"Fetching {url}")
            response = self.session.get(url, timeout=10)
            response.raise_for_status()

            # Respect crawl delay
            time.sleep(self.crawl_delay)

            return BeautifulSoup(response.text, "html.parser")

        except requests.exceptions.RequestException as e:
            self.logger.error(f"Error fetching {url}: {e}")
            return None

In [ ]:
# Use our respectful scraper to fetch products
scraper = RespectfulScraper("http://books.toscrape.com", default_delay=1)

soup = scraper.get("http://books.toscrape.com/catalogue/page-1.html")

if soup:
    products = soup.find_all("article", class_="product_pod")
    print(f"Successfully scraped {len(products)} products (respectfully!)")

Notice how the `RespectfulScraper` class encapsulates all of the ethical practices we discussed:

- Checks robots.txt before every request &check;
- Identifies itself with a clear User-Agent string &check; 
- Respects crawl delays &check;
- Handles errors gracefully with logging &check;
- Uses a session for efficient connection reuse &check; 

> 💼 **Assessment connection:** The rubric specifically rewards "strong awareness of the regulatory/legal/ethical implications" and "robust error handling." This class demonstrates both. In your coursework, explain *why* you made each design decision.

---

# 🏫 Part 4: Bringing It Together

*Approximate time: 20 minutes*

## 4.1 Choosing the Right Tool

There are three main Python libraries for web scraping, each with different strengths:

| Tool | Best For | Handles JavaScript? | Complexity |
|------|----------|-------------------|------------|
| **BeautifulSoup** + requests | Simple, static pages | No | Low |
| **Scrapy** | Large-scale crawling, multiple pages | No (without middleware) | Medium |
| **Selenium** | JavaScript-rendered pages, interactive sites | Yes | High |

**How to decide:**
1. First, try `requests` + BeautifulSoup — it is the simplest approach and works for most static sites
2. If you need to scrape many pages efficiently or follow complex link structures, consider Scrapy
3. If the page content is loaded by JavaScript (you see a blank page in the HTML), use Selenium

> 💼 **Assessment connection:** The marking criteria ask you to discuss "the various tools available to data scientists" and explain your choice. Your technical document should justify *why* you chose a particular tool for your specific use case.

## 4.2 From Scraping to Analysis: A Preview

In Topic 2, you built a text preprocessing pipeline. Web scraping is how you *collect* the raw text that pipeline processes. Let's see how the two connect.

Imagine you have scraped product descriptions from a competitor's website. You can apply the same preprocessing steps from Topic 2 to clean and analyse that text:

In [ ]:
# Simulate scraped product descriptions (as if we had scraped review text)
scraped_descriptions = [
    "A Light in the Attic: A beautiful collection of poems that will DELIGHT readers of all ages!!!",
    "Tipping the Velvet: An extraordinary debut novel... absolutely captivating & unputdownable.",
    "Soumission: A controversial, thought-provoking novel. NOT for the faint-hearted.",
    "Sharp Objects: Dark, twisted & disturbing. The writing is superb but the subject matter is grim.",
    "Sapiens: BRILLIANT overview of human history. Every chapter is fascinating!!! A must-read.",
]

# Quick text cleaning - reusing concepts from Topic 2
def basic_clean(text: str) -> str:
    """Apply basic text cleaning to scraped text."""
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

for desc in scraped_descriptions[:3]:
    cleaned = basic_clean(desc)
    print(f"Raw:     {desc[:70]}...")
    print(f"Cleaned: {cleaned[:70]}...")
    print()

This is a simplified version of what you built in Topic 2. In Part 6 (independent study), you will build a complete end-to-end pipeline that scrapes real web data, preprocesses it, and runs sentiment analysis — exactly the kind of workflow the assessment expects.

## 4.3 What's Coming Next

The skills you have built today form the foundation for the rest of the module:

- **Topic 4 (APIs)** — many websites offer APIs as a structured alternative to scraping. You will learn when to scrape and when to use an API.
- **Topic 6 (Text Analytics)** — you will apply NLP techniques to the kind of text data you have learned to collect today.
- **Your Assessment** — combines scraping (or API collection) with preprocessing and analysis to deliver business value.

---

---

# 📖 Part 5: Dynamic Websites with Selenium

*Independent study — work through this section in your own time*

## 5.1 Why Some Websites Need a Browser

The `requests` library fetches the *initial* HTML from a server. But many modern websites use JavaScript to load content *after* the page has loaded in a browser. If you fetch such a page with `requests`, you get an empty shell.

Common examples include:
- Social media feeds (Twitter/X, BlueSky, LinkedIn)
- Single-page applications (SPAs) built with React, Angular, or Vue
- Pages with infinite scroll or "Load More" buttons
- Content behind cookie consent dialogs

**Selenium** solves this by automating a real web browser. It launches Chrome (or Firefox), navigates to the page, waits for JavaScript to execute, and then gives you the fully-rendered HTML.

## 5.2 Setting Up Selenium

Selenium requires a browser driver. The `webdriver-manager` library handles this automatically:

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import WebDriverException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager

In [ ]:
def create_headless_browser() -> webdriver.Chrome:
    """Create a headless Chrome browser instance."""
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")           # No visible browser window
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")

    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)

### Headless Mode

The `--headless` flag tells Chrome to run without a visible window. This is essential for:
- Running scrapers on servers without a display
- Automated pipelines and CI/CD
- Being less resource-intensive

## 5.3 Scraping a JavaScript-Rendered Page

Let's use Selenium to scrape [Quotes to Scrape (JS version)](http://quotes.toscrape.com/js/), a practice site that renders its content with JavaScript. If you try this with `requests`, you will get an empty page:

In [ ]:
# First, let's prove that requests can't see the JavaScript-rendered content
response = requests.get("http://quotes.toscrape.com/js/")
soup_static = BeautifulSoup(response.text, "html.parser")

quotes_static = soup_static.find_all("div", class_="quote")
print(f"Quotes found with requests: {len(quotes_static)}")
print("(The content is loaded by JavaScript — requests can't see it)")

NameError: name 'requests' is not defined

In [ ]:
# Now let's use Selenium to render the JavaScript
try:
    browser = create_headless_browser()
    browser.get("http://quotes.toscrape.com/js/")

    # Wait for quotes to appear (up to 10 seconds)
    WebDriverWait(browser, 10).until(
        EC.presence_of_element_located((By.CLASS_NAME, "quote"))
    )

    # Get the fully-rendered HTML
    rendered_html = browser.page_source
    soup_dynamic = BeautifulSoup(rendered_html, "html.parser")

    quotes = soup_dynamic.find_all("div", class_="quote")
    print(f"Quotes found with Selenium: {len(quotes)}")

    # Extract the quote data
    quote_data = []
    for quote in quotes:
        text = quote.find("span", class_="text").get_text()
        author = quote.find("small", class_="author").get_text()
        tags = [tag.get_text() for tag in quote.find_all("a", class_="tag")]
        quote_data.append({
            "text": text,
            "author": author,
            "tags": ", ".join(tags)
        })

    df_quotes = pd.DataFrame(quote_data)
    df_quotes.head()

finally:
    # Always close the browser to free resources
    browser.quit()

### Key Selenium Patterns

There are a few important patterns to remember when using Selenium:

**Always use explicit waits** rather than `time.sleep()`. Explicit waits are more reliable and faster:
```python
# Good — waits until the element actually appears
WebDriverWait(browser, 10).until(
    EC.presence_of_element_located((By.CLASS_NAME, "quote"))
)

# Less good — waits a fixed time regardless
time.sleep(5)
```

**Always close the browser** when you are finished. Use a `try`/`finally` block or a context manager to ensure the browser is closed even if an error occurs.

**Combine Selenium with BeautifulSoup** for parsing. Selenium is great at rendering pages, but BeautifulSoup is better at navigating the HTML tree. Use Selenium to get the rendered HTML, then parse it with BeautifulSoup.

> 💡 **When to use Selenium vs BeautifulSoup:** If you view the page source (Ctrl+U) and the content is missing, you need Selenium. If the content is in the source, BeautifulSoup is sufficient and much faster.

---

---

# 📖 Part 6: From Scraping to Insight

*Independent study — this section connects web scraping to the preprocessing pipeline from Topic 2*

## 6.1 The Complete Pipeline

Your module assessment asks you to build an automated process that collects unstructured data and converts it into a structured format to add business value. Here is what that complete pipeline looks like:

```text
┌─────────────┐   ┌──────────────┐   ┌──────────────┐   ┌──────────────┐
│  1. Scrape  │ → │  2. Clean &  │ → │  3. Analyse  │ → │  4. Present  │
│  Web Data   │   │  Preprocess  │   │  (NLP, etc.) │   │  Insights    │
└─────────────┘   └──────────────┘   └──────────────┘   └──────────────┘
```

Let's build this pipeline end-to-end, using the book descriptions we scraped earlier.

## 6.2 Step 1: Scrape Product Descriptions

First, let's scrape the detailed descriptions from individual product pages:

In [ ]:
def scrape_book_details(book_url: str) -> Optional[dict]:
    """Scrape detailed information from a single book's page."""
    try:
        response = requests.get(book_url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        # Extract the product description
        description_tag = soup.find("div", id="product_description")
        description = ""
        if description_tag:
            description = description_tag.find_next_sibling("p").get_text(strip=True)

        # Extract the title
        title = soup.find("h1").get_text(strip=True)

        return {"title": title, "description": description}

    except Exception as e:
        print(f"Error scraping {book_url}: {e}")
        return None

In [ ]:
# Scrape descriptions for the first 5 books
base_url = "http://books.toscrape.com/catalogue/"

# Get links from the first page
soup = BeautifulSoup(requests.get("http://books.toscrape.com/").text, "html.parser")
products = soup.find_all("article", class_="product_pod")

book_details = []
for product in products[:5]:  # Limit to 5 for demonstration
    relative_link = product.find("h3").find("a")["href"]
    full_url = urljoin("http://books.toscrape.com/", relative_link)

    detail = scrape_book_details(full_url)
    if detail:
        book_details.append(detail)
        print(f"✓ {detail['title'][:50]}")

    time.sleep(1)  # Respectful delay

df_details = pd.DataFrame(book_details)
df_details

## 6.3 Step 2: Preprocess the Text

Now we apply the preprocessing techniques from Topic 2. This is the bridge between data collection and analysis:

In [ ]:
def preprocess_text(text: str) -> str:
    """
    Clean scraped text for analysis.
    Applies the preprocessing pipeline from Topic 2.
    """
    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    # Remove HTML entities and tags
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"&\w+;", "", text)

    # Remove punctuation (keep alphanumeric and spaces)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)

    # Normalise whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Apply preprocessing to the scraped descriptions
df_details["cleaned"] = df_details["description"].apply(preprocess_text)

# Compare raw vs cleaned
for _, row in df_details.head(3).iterrows():
    print(f"Title:   {row['title']}")
    print(f"Raw:     {row['description'][:80]}...")
    print(f"Cleaned: {row['cleaned'][:80]}...")
    print()

## 6.4 Step 3: Simple Sentiment Analysis

Finally, let's apply a quick sentiment analysis to the cleaned descriptions — the same approach you explored in Topic 2 with TextBlob:

In [ ]:
%pip install --quiet textblob

In [ ]:
from textblob import TextBlob

# Calculate sentiment for each description
df_details["sentiment_raw"] = df_details["description"].apply(
    lambda x: TextBlob(x).sentiment.polarity
)
df_details["sentiment_cleaned"] = df_details["cleaned"].apply(
    lambda x: TextBlob(x).sentiment.polarity
)

# Display results
result = df_details[["title", "sentiment_raw", "sentiment_cleaned"]].copy()
result.columns = ["Title", "Raw Sentiment", "Cleaned Sentiment"]
result

Notice how the sentiment scores differ between raw and preprocessed text. This is the same trade-off you explored in Topic 2 — preprocessing can sometimes remove signals (like punctuation and capitalisation) that sentiment tools use.

> 💼 **Assessment connection:** This end-to-end pipeline — scrape → preprocess → analyse — is exactly what the summative assessment requires. Your technical document should explain each step and justify your design decisions.

---

---

# 📝 Exercises

## Consolidation Exercises

These exercises reinforce the core concepts from the session. They are recommended for all learners.

### Exercise 1: Scrape a Different Page

Modify the `scrape_page()` function to also extract the **book's URL** (the `href` attribute from the title link). Scrape pages 1–5 and save the results to a CSV file.

*Hint:* Use `df.to_csv("books.csv", index=False)` to save a DataFrame.

### Exercise 2: robots.txt Analysis

Write a function that takes a URL and returns a summary of the site's robots.txt rules, including:
- Whether the homepage is accessible
- The crawl delay (if specified)
- A list of disallowed paths

Test it on at least 3 different websites relevant to your workplace.

### Exercise 3: Error Handling

The current `RespectfulScraper.get()` method catches all request exceptions. Extend it to:
- Retry failed requests up to 3 times with exponential backoff
- Log the specific error type (timeout, connection error, HTTP error)
- Return `None` only after all retries are exhausted

## Extension Exercises

These exercises go beyond the core material for learners who want to stretch further.

### Exercise 4: Workplace Application

Identify a website relevant to your organisation that could provide business value if scraped. Write a short analysis (around 300 words) covering:
1. What data would you extract, and what business question would it answer?
2. Which tool (BeautifulSoup, Scrapy, or Selenium) would you choose, and why?
3. What legal and ethical considerations would you need to address?
4. How would you ensure your scraping is respectful and sustainable?

> This exercise directly prepares you for the technical document in your summative assessment.

### Exercise 5: End-to-End Pipeline

Build a complete pipeline that:
1. Scrapes data from a website of your choice
2. Cleans and preprocesses the text
3. Performs a simple analysis (e.g., word frequency, sentiment)
4. Saves the results to a structured format (CSV or JSON)

Document each step with clear comments explaining your decisions.

---

## Further Reading

- [Automate the Boring Stuff — Chapter 12: Web Scraping](https://automatetheboringstuff.com/2e/chapter12/)
- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Selenium with Python Documentation](https://selenium-python.readthedocs.io/)
- [ICO Guidance on Web Scraping](https://ico.org.uk/) — UK data protection authority
- [Web Scraping and the Computer Misuse Act](https://www.legislation.gov.uk/ukpga/1990/18/contents)